# HrudayAI — End-to-End Reviewer Demo

**Pipeline:** Audio Upload (2 recordings) → Phase 1 Transcription (`faster-whisper-small`) → Phase 2 / 2.5 Symptom Extraction & Validation → MedPredict Symptom Adapter → Standalone XGBoost Inference (131 features) → Plain-Text Top-3 Candidate Conditions

- **Repositories:** `itsakshatsinghh/HrudayAI` + `itsakshatsinghh/MedPredict` (public GitHub repos; zero API keys or paid services)
- **Runtime:** Google Colab CPU-compatible
- **Disclaimer:** Educational prototype only; model outputs are candidate conditions, not clinical diagnoses.

## Section 1 — Setup

Clones or updates `HrudayAI` and `MedPredict` from public GitHub repositories, configures Python paths, and installs only the required open-source packages.

In [ ]:
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
HRUDAY_URL = "https://github.com/itsakshatsinghh/HrudayAI.git"
MEDPREDICT_URL = "https://github.com/itsakshatsinghh/MedPredict.git"

if IN_COLAB:
    COLAB_HRUDAY = Path("/content/HrudayAI")
    freshly_cloned = False
    if not COLAB_HRUDAY.exists():
        print("Cloning HrudayAI repository...")
        subprocess.run(["git", "clone", "--branch", "main", HRUDAY_URL, str(COLAB_HRUDAY)], check=True)
        freshly_cloned = True
    if str(COLAB_HRUDAY) not in sys.path:
        sys.path.insert(0, str(COLAB_HRUDAY))

    from shared.colab_bootstrap import bootstrap_repo
    bootstrap_info = bootstrap_repo(pull_latest=not freshly_cloned)
    REPO_ROOT = bootstrap_info["repo_root"]

    MEDPREDICT_ROOT = Path("/content/MedPredict")
    if not MEDPREDICT_ROOT.exists():
        print("Cloning MedPredict model repository...")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", MEDPREDICT_URL, str(MEDPREDICT_ROOT)], check=True)
    else:
        print("MedPredict repository already present; pulling latest main...")
        subprocess.run(["git", "-C", str(MEDPREDICT_ROOT), "pull", "--ff-only", "origin", "main"], check=False)
else:
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "shared" / "colab_bootstrap.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            break
    from shared.colab_bootstrap import bootstrap_repo
    bootstrap_info = bootstrap_repo(pull_latest=False)
    REPO_ROOT = bootstrap_info["repo_root"]
    MEDPREDICT_ROOT = REPO_ROOT.parent / "MedPredict"

# Install only necessary open-source dependencies in Colab
if IN_COLAB:
    print("Installing Phase 1, Phase 2, and Phase 3 dependencies...")
    req_files = [
        REPO_ROOT / "phase1_voice" / "requirements.txt",
        REPO_ROOT / "phase2_symptom_extraction" / "requirements.txt",
        REPO_ROOT / "phase2_prediction" / "requirements.txt",
    ]
    for req in req_files:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])
    importlib.invalidate_caches()

MEDPREDICT_DIR = MEDPREDICT_ROOT / "disease predictor"
assert (MEDPREDICT_DIR / "model.json").exists(), f"Missing model.json at {MEDPREDICT_DIR}"
assert (MEDPREDICT_DIR / "label_encoder.pkl").exists(), f"Missing label_encoder.pkl at {MEDPREDICT_DIR}"

from phase1_voice.src import resolve_device
device, compute_type = resolve_device("auto")
print(f"Setup complete | HrudayAI={REPO_ROOT} | MedPredict={MEDPREDICT_DIR} | Device={device} ({compute_type}) | ffmpeg={shutil.which('ffmpeg') is not None}")

## Section 2 — Upload Audio (Two Recordings)

Upload two audio recordings (e.g., Audio 1 in Hindi and Audio 2 in English; `.wav`, `.mp3`, `.m4a`, `.flac`, `.ogg`). Each file is preprocessed independently to 16 kHz mono.

In [ ]:
from IPython.display import Audio, display
from phase1_voice.src import get_audio_metadata, load_audio

AUDIO_PATHS = {}

if IN_COLAB:
    from google.colab import files  # type: ignore

    for slot in ["Audio 1", "Audio 2"]:
        print(f"\nPlease upload {slot} (supported: .wav, .mp3, .m4a, .flac, .ogg):")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError(f"No file uploaded for {slot}. Please re-run this cell and upload two audio files.")
        fname = next(iter(uploaded.keys()))
        AUDIO_PATHS[slot] = Path(fname).resolve()
else:
    # Local optional paths for non-Colab testing (set manually if testing locally)
    AUDIO_PATHS = {
        "Audio 1": None,
        "Audio 2": None,
    }
    print("Non-Colab mode: set AUDIO_PATHS['Audio 1'] and AUDIO_PATHS['Audio 2'] if testing with local files.")

PREPROCESSED_AUDIO = {}
for slot, path in AUDIO_PATHS.items():
    if path is None:
        print(f"[{slot}] Skipped (no path provided).")
        continue
    print(f"[{slot}] Loading and preprocessing {path.name}...")
    waveform, sr = load_audio(path, target_sr=16000, mono=True, normalize=True)
    meta = get_audio_metadata(waveform, sr)
    PREPROCESSED_AUDIO[slot] = (waveform, sr, meta)
    print(f"[{slot}] Ready -> duration={meta['duration_seconds']}s, sample_rate={meta['sample_rate_hz']}Hz")
    display(Audio(waveform, rate=sr))

## Section 3 — Transcription (Phase 1)

Loads `faster-whisper-small` once (cached in runtime memory) and transcribes each uploaded audio clip with automatic Hindi/English language detection.

In [ ]:
import time
from phase1_voice.src import load_asr_model, transcribe_audio

TRANSCRIPTION_RESULTS = {}

if not PREPROCESSED_AUDIO:
    print("No preprocessed audio loaded. Upload two files in Section 2 to run transcription.")
else:
    print("Loading open-source faster-whisper-small model (cached for this runtime)...")
    ASR_BUNDLE = load_asr_model(model_key="faster_whisper_small", device="auto")

    for slot, (waveform, sr, _meta) in PREPROCESSED_AUDIO.items():
        t0 = time.perf_counter()
        print(f"Transcribing {slot}...")
        result = transcribe_audio(
            waveform=waveform,
            sample_rate=sr,
            model_bundle=ASR_BUNDLE,
            language=None,  # Auto-detect Hindi / English
        )
        elapsed = round(time.perf_counter() - t0, 2)
        TRANSCRIPTION_RESULTS[slot] = result
        print(f"\n--- {slot} Transcription ({elapsed}s) ---")
        print(f"Detected Language : {result['detected_language']} (prob={result['language_probability']})")
        print(f"Transcript        : {result['text']}")

## Section 4 — Symptom Extraction (Phase 2 & 2.5)

Reuses `phase2_symptom_extraction` to extract canonical symptom names, original phrases, duration, severity, negation, temporal status, and certainty.

In [ ]:
import pandas as pd
from phase2_symptom_extraction.src import extract_from_phase1_payload

EXTRACTION_RESULTS = {}

if not TRANSCRIPTION_RESULTS:
    print("No transcription results available. Run Section 3 first.")
else:
    for slot, p1_res in TRANSCRIPTION_RESULTS.items():
        p2_res = extract_from_phase1_payload(p1_res)
        EXTRACTION_RESULTS[slot] = p2_res
        print(f"\n=== {slot} Extracted Symptoms (language={p2_res['language']}) ===")
        if p2_res["symptoms"]:
            df_sym = pd.DataFrame(p2_res["symptoms"])[
                ["canonical_name", "original_term", "duration", "severity", "negated", "temporal_status", "certainty"]
            ]
            display(df_sym)
        else:
            print("No known symptoms extracted from this transcript.")

## Section 5 — Model Adapter (Phase 2.5 → MedPredict 131 Features)

Filters strictly for `temporal_status='current'`, `certainty='confirmed'`, and `negated=False`, preserves MedPredict's exact 131-feature order, and surfaces excluded, unmatched, and ambiguous symptoms (never guessing generic `fever` or `weakness`).

In [ ]:
from phase2_prediction.src import adapt_phase2_to_medpredict, load_medpredict_artifacts

print("Loading MedPredict standalone XGBoost model.json and label_encoder.pkl...")
MEDPREDICT_BUNDLE = load_medpredict_artifacts(medpredict_dir=MEDPREDICT_DIR, repo_root=REPO_ROOT)
print(f"Loaded MedPredict: {len(MEDPREDICT_BUNDLE['feature_names'])} features, {len(MEDPREDICT_BUNDLE['label_encoder'].classes_)} disease classes.")

ADAPTER_AUDITS = {}
if not EXTRACTION_RESULTS:
    print("No extracted symptoms available. Run Section 4 first.")
else:
    for slot, p2_res in EXTRACTION_RESULTS.items():
        audit = adapt_phase2_to_medpredict(p2_res, model_features=MEDPREDICT_BUNDLE["feature_names"])
        ADAPTER_AUDITS[slot] = audit
        print(f"\n=== {slot} Adapter Audit ===")
        print(f"Can Predict              : {audit['can_predict']}")
        print(f"Matched Model Features   : {audit['matched_features']}")
        print(f"Excluded (hist/neg/unc)  : {audit['excluded_symptoms']}")
        print(f"Needs Clarification (amb): {audit['ambiguous_symptoms']}")
        print(f"Unmatched Symptoms       : {audit['unmatched_symptoms']}")

## Section 6 — Disease Prediction (Local XGBoost Inference)

Runs `predict_proba()` on the exact 131-feature vector when at least 1 valid feature matched; otherwise skips model inference and shows a clarification message.

In [ ]:
from phase2_prediction.src import predict_top_conditions

PREDICTION_RESULTS = {}
if not EXTRACTION_RESULTS:
    print("No extraction results available. Run Section 4 and Section 5 first.")
else:
    for slot, p2_res in EXTRACTION_RESULTS.items():
        pred_res = predict_top_conditions(p2_res, model_bundle=MEDPREDICT_BUNDLE, top_k=3)
        PREDICTION_RESULTS[slot] = pred_res
        print(f"\n=== {slot} Top-3 Candidate Conditions ===")
        if pred_res["status"] == "predicted":
            display(pd.DataFrame(pred_res["predictions"]))
        else:
            print(f"Model skipped: {pred_res['message']}")

## Section 7 — Final Reviewer Output & Side-by-Side Comparison

In [ ]:
from phase2_prediction.src import format_plain_text_report

if not PREDICTION_RESULTS:
    print("No prediction results to display. Complete Sections 2-6 with two uploaded audio files.")
else:
    for slot in ["Audio 1", "Audio 2"]:
        if slot in PREDICTION_RESULTS:
            report = format_plain_text_report(
                audio_label=slot,
                phase1_result=TRANSCRIPTION_RESULTS[slot],
                phase2_result=EXTRACTION_RESULTS[slot],
                prediction_result=PREDICTION_RESULTS[slot],
            )
            print(report)
            print("\n" + "-" * 72 + "\n")

    # Compact comparison table across Audio 1 and Audio 2
    comparison_rows = []
    for slot in ["Audio 1", "Audio 2"]:
        if slot not in PREDICTION_RESULTS:
            continue
        p1 = TRANSCRIPTION_RESULTS[slot]
        p2 = EXTRACTION_RESULTS[slot]
        pr = PREDICTION_RESULTS[slot]
        audit = pr["adapter_audit"]
        top_str = (
            "; ".join(f"{p['condition']} ({p['score_percent']:.2f}%)" for p in pr["predictions"])
            if pr["status"] == "predicted"
            else "Clarification required (not run)"
        )
        comparison_rows.append(
            {
                "Recording": slot,
                "Language": p1.get("detected_language"),
                "Extracted Symptoms": ", ".join(s["canonical_name"] for s in p2.get("symptoms", [])),
                "Matched Features": ", ".join(audit["matched_features"]),
                "Clarification / Unmatched": ", ".join(
                    [a["canonical_name"] for a in audit["ambiguous_symptoms"]]
                    + [u["canonical_name"] for u in audit["unmatched_symptoms"]]
                ),
                "Top-3 Candidate Conditions": top_str,
            }
        )

    print("=== Compact Comparison (Audio 1 vs Audio 2) ===")
    display(pd.DataFrame(comparison_rows))